In [1]:
# =========================================
# Cell 1 — Install deps (Colab-safe)
# =========================================
!pip -q install -U transformers accelerate bitsandbytes datasets huggingface_hub tqdm pandas pyarrow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 155.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 58.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [4]:
# =========================================
# Cell 2 — HF login (for gated Llama)
# =========================================
import os, getpass
from huggingface_hub import login

os.environ["HF_TOKEN"] = getpass.getpass("HF token (hidden): ").strip()
assert os.environ["HF_TOKEN"], "HF token is required"

login(token=os.environ["HF_TOKEN"])
print("✅ HF login done")


HF token (hidden): ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HF login done


In [8]:
# =========================================
# Cell 3 — Load FiQA-SA (Hugging Face)
# =========================================
from datasets import load_dataset

FIQASA_REPO = "TheFinAI/flare-fiqasa"   # if this errors, try "TheFinAI/fiqa-sentiment" or paste your repo name
SPLIT = "test"

ds = load_dataset(FIQASA_REPO, split=SPLIT)
print(ds)
print("Columns:", ds.column_names)
print("Example:", ds[0])

# True label order if ClassLabel exists
if "label" in ds.features and hasattr(ds.features["label"], "names"):
    DS_LABELS = [x.strip().lower() for x in ds.features["label"].names]
    print("✅ Dataset label order:", DS_LABELS)
else:
    # FiQA-SA is typically {negative, neutral, positive}
    DS_LABELS = ["negative", "neutral", "positive"]
    print("⚠️ No ClassLabel found. Using fallback:", DS_LABELS)


README.md:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 235
})
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Example: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}
⚠️ No ClassLabel found. Using fallback: ['negative', 'neutral', 'positive']


In [9]:
# =========================================
# Cell 4 — Load Llama-3.1-8B-Instruct (4-bit)
# =========================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ["HF_TOKEN"], use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=os.environ["HF_TOKEN"],
    device_map="auto",
    quantization_config=bnb,
)
model.eval()
print("✅ Model loaded:", MODEL_NAME)


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✅ Model loaded: meta-llama/Llama-3.1-8B-Instruct


In [10]:
# =========================================
# Cell 5 — Eval loop (resume supported) using dataset label order
# =========================================
import os, json, re, random
from tqdm import tqdm

def get_text(ex: dict) -> str:
    # common FiQA fields
    for k in ["text", "sentence", "content", "input", "headline", "title", "question", "context"]:
        if k in ex and ex[k] is not None and str(ex[k]).strip():
            return str(ex[k])
    # sometimes text is split (headline + body)
    if "headline" in ex and "body" in ex:
        return f"{ex['headline']}\n{ex['body']}"
    raise KeyError(f"No text field found. Keys={list(ex.keys())}")

def get_gold(ex: dict) -> int:
    for k in ["label", "gold", "sentiment"]:
        if k in ex and ex[k] is not None:
            v = ex[k]
            if isinstance(v, str):
                v2 = v.strip().lower()
                if v2 in DS_LABELS:
                    return DS_LABELS.index(v2)
                # aliases
                alias = {"pos":"positive","neg":"negative","neu":"neutral"}
                if v2 in alias and alias[v2] in DS_LABELS:
                    return DS_LABELS.index(alias[v2])
                raise ValueError(f"Unknown label string: {v}")
            return int(v)
    raise KeyError(f"No label field found. Keys={list(ex.keys())}")

def parse_pred(raw: str):
    if not raw:
        return None
    s = raw.strip().lower()
    for lab in DS_LABELS:
        if s == lab:
            return lab
        if re.search(rf"\b{re.escape(lab)}\b", s):
            return lab
    alias = {"pos":"positive","neg":"negative","neu":"neutral"}
    for a, full in alias.items():
        if re.search(rf"\b{a}\b", s) and full in DS_LABELS:
            return full
    return None

def llama_classify(text: str, max_new_tokens=8) -> str:
    messages = [
        {"role": "system", "content": "You are a financial sentiment classifier."},
        {"role": "user", "content": (
            "Classify the sentiment of the following financial text.\n"
            f"Respond with exactly ONE label from: {', '.join(DS_LABELS)}.\n\n"
            f"Text: {text}\n"
            "Label:"
        )}
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        out = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0, input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

# ---- run config ----
SEED = 42
MAX_SAMPLES = None   # None = full split; set e.g. 200 for quick test
SHUFFLE = True

random.seed(SEED)
os.makedirs("text_responses", exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

run_id = f"llama31_8b_fiqasa_{FIQASA_REPO.replace('/','_')}_{SPLIT}"
raw_path = f"text_responses/{run_id}.jsonl"

# resume
done = set()
if os.path.exists(raw_path):
    with open(raw_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["row_index"])
            except:
                pass
print("Already completed:", len(done))

idxs = list(range(len(ds)))
if SHUFFLE:
    random.shuffle(idxs)
if MAX_SAMPLES is not None:
    idxs = idxs[:MAX_SAMPLES]

with open(raw_path, "a", encoding="utf-8") as f:
    for i in tqdm(idxs, desc="Evaluating"):
        if i in done:
            continue
        ex = ds[i]
        text = get_text(ex)
        gold = get_gold(ex)

        raw = llama_classify(text)
        pred_label = parse_pred(raw)
        pred = DS_LABELS.index(pred_label) if pred_label in DS_LABELS else -1

        rec = {
            "row_index": i,
            "text": text,
            "gold": gold,
            "predicted_index": pred,
            "predicted_label": pred_label,
            "raw": raw,
            "correct": (pred == gold),
            "ds_labels": DS_LABELS,
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("✅ Saved:", raw_path)


Already completed: 0


Evaluating:   0%|          | 0/235 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Evaluating: 100%|██████████| 235/235 [00:44<00:00,  5.25it/s]

✅ Saved: text_responses/llama31_8b_fiqasa_TheFinAI_flare-fiqasa_test.jsonl


In [11]:
# =========================================
# Cell 6 — Metrics (pure Python)
# =========================================
import json, os

rows = []
with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            rows.append(json.loads(line))
        except:
            pass

gold = [r["gold"] for r in rows]
pred = [r["predicted_index"] for r in rows]

valid = [(g, p) for g, p in zip(gold, pred) if isinstance(p, int) and p >= 0]
gold_v = [g for g, _ in valid]
pred_v = [p for _, p in valid]

def accuracy(y_true, y_pred):
    return sum(int(a == b) for a, b in zip(y_true, y_pred)) / max(1, len(y_true))

def macro_f1(y_true, y_pred, classes):
    f1s = []
    for c in classes:
        tp = sum((yt == c and yp == c) for yt, yp in zip(y_true, y_pred))
        fp = sum((yt != c and yp == c) for yt, yp in zip(y_true, y_pred))
        fn = sum((yt == c and yp != c) for yt, yp in zip(y_true, y_pred))
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        f1s.append(f1)
    return sum(f1s) / len(f1s) if f1s else None

classes = sorted(set(gold))
cm = [[0 for _ in classes] for __ in classes]
for g, p in valid:
    cm[g][p] += 1

metrics = {
    "run_id": run_id,
    "model": MODEL_NAME,
    "dataset": FIQASA_REPO,
    "split": SPLIT,
    "ds_labels": DS_LABELS,
    "n_total": len(rows),
    "n_valid": len(valid),
    "accuracy_all": accuracy(gold, pred),
    "accuracy_valid": accuracy(gold_v, pred_v) if valid else None,
    "macro_f1_valid": macro_f1(gold_v, pred_v, classes) if valid else None,
    "confusion_matrix_classes": classes,
    "confusion_matrix": cm,
}

print(metrics)

out_path = f"evaluation/{run_id}_metrics.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("✅ Metrics saved:", out_path)


{'run_id': 'llama31_8b_fiqasa_TheFinAI_flare-fiqasa_test', 'model': 'meta-llama/Llama-3.1-8B-Instruct', 'dataset': 'TheFinAI/flare-fiqasa', 'split': 'test', 'ds_labels': ['negative', 'neutral', 'positive'], 'n_total': 235, 'n_valid': 235, 'accuracy_all': 0.3617021276595745, 'accuracy_valid': 0.3617021276595745, 'macro_f1_valid': 0.36291341508732816, 'confusion_matrix_classes': [0, 1, 2], 'confusion_matrix': [[60, 11, 5], [9, 21, 111], [5, 9, 4]]}
✅ Metrics saved: evaluation/llama31_8b_fiqasa_TheFinAI_flare-fiqasa_test_metrics.json
